In [3]:
import pandas as pd
import numpy as np
import datetime
from collections import defaultdict
from pathlib import Path
import requests
import os
import re
import io

import warnings
warnings.filterwarnings('ignore')

In [29]:
def get_23_quantiles_interp(group, max_bin_width=0.05):
    """
    convert binned forecast into quantiles format by interpolating across the bins
    
    """
    quantiles = [0.01, 0.025] + list(np.arange(0.05, 0.96, 0.05)) + [0.975, 0.99]
    quantiles = [round(q, 3) for q in quantiles]

    group = group.sort_values('bin_start_incl').reset_index(drop=True)
    bin_widths = group['bin_end_notincl'] - group['bin_start_incl']

    out = []
    for q in quantiles:
        idx_list = group.index[group['cum_val'] >= q]

        if len(idx_list) == 0:
            out.append(group.iloc[-1]['bin_end_notincl'])
            continue

        idx = idx_list[0]
        row = group.loc[idx]
        prev_cum = group.loc[idx - 1, 'cum_val'] if idx > 0 else 0.0
        bin_width = bin_widths.iloc[idx]

        if bin_width <= max_bin_width:
            if row['cum_val'] > prev_cum:
                frac = (q - prev_cum) / (row['cum_val'] - prev_cum)
                value = row['bin_start_incl'] + frac * bin_width
            else:
                value = row['bin_start_incl']
        else:
            value = _extrapolate_from_previous_bin(
                group, idx, q, bin_widths, max_bin_width
            )

        out.append(value)

    return pd.DataFrame({'quantile': quantiles, 'value': out})


def _extrapolate_from_previous_bin(group, idx, q, bin_widths, max_bin_width):
    wide_bin_start = group.loc[idx, 'bin_start_incl']
    cum_at_wide_start = group.loc[idx - 1, 'cum_val'] if idx > 0 else 0.0

    prev_idx = idx - 1
    while prev_idx >= 0 and bin_widths.iloc[prev_idx] > max_bin_width:
        prev_idx -= 1

    if prev_idx <= 0:
        return wide_bin_start

    prev_row = group.loc[prev_idx]
    cum_before_prev = group.loc[prev_idx - 1, 'cum_val'] if prev_idx > 0 else 0.0
    prev_width = prev_row['bin_end_notincl'] - prev_row['bin_start_incl']
    mass_in_prev = prev_row['cum_val'] - cum_before_prev

    if mass_in_prev <= 0 or prev_width <= 0:
        return wide_bin_start

    slope = mass_in_prev / prev_width
    value = wide_bin_start + (q - cum_at_wide_start) / slope

    return max(value, wide_bin_start)



In [28]:
# get all team forecasts 
forecasts1516_all= pd.read_csv('../dat/ILI team forecasts/ILIforecasts_teams_2015-16.csv.gz')
forecasts1617_all= pd.read_csv('../dat/ILI team forecasts/ILIforecasts_teams_2016-17.csv.gz')
forecasts1718_all= pd.read_csv('../dat/ILI team forecasts/ILIforecasts_teams_2017-18.csv.gz')
forecasts1819_all= pd.read_csv('../dat/ILI team forecasts/ILIforecasts_teams_2018-19.csv.gz')
forecasts1920_all= pd.read_csv('../dat/ILI team forecasts/ILIforecasts_teams_2019-20.csv.gz')

# Convert to quantiles

In [30]:
df = forecasts1516_all.copy()
df = df.sort_values(['Model','location','forecast_date', 'horizon', 'bin_start_incl','submission_date']).copy()
df['cum_val'] = df.groupby(['Model','location','horizon','forecast_date','submission_date'])['value'].cumsum()


result1516 = (df.groupby(['Model',"location", "horizon", "forecast_date", 'season', 'target_end_date',
                      'submission_date']).apply(get_23_quantiles_interp).reset_index()).drop(columns='level_7')


In [35]:
result1516.to_csv('../dat/ILI team forecasts/ILIforecasts_teams_quantiles_2015-16.csv.gz', index=False, compression='gzip')

In [32]:
df = forecasts1617_all.copy()
df = df.sort_values(['Model','location','forecast_date', 'horizon', 'bin_start_incl','submission_date']).copy()
df['cum_val'] = df.groupby(['Model','location','horizon','forecast_date','submission_date'])['value'].cumsum()


result1617 = (df.groupby(['Model',"location", "horizon", "forecast_date", 'season', 'target_end_date',
                      'submission_date']).apply(get_23_quantiles_interp).reset_index()).drop(columns='level_7')


In [36]:
result1617.to_csv('../dat/ILI team forecasts/ILIforecasts_teams_quantiles_2016-17.csv.gz', index=False, compression='gzip')

In [37]:
df = forecasts1718_all.copy()
df = df.sort_values(['Model','location','forecast_date', 'horizon', 'bin_start_incl','submission_date']).copy()
df['cum_val'] = df.groupby(['Model','location','horizon','forecast_date','submission_date'])['value'].cumsum()


result1718 = (df.groupby(['Model',"location", "horizon", "forecast_date", 'season', 'target_end_date',
                      'submission_date']).apply(get_23_quantiles_interp).reset_index()).drop(columns='level_7')


In [38]:
result1718.to_csv('../dat/ILI team forecasts/ILIforecasts_teams_quantiles_2017-18.csv.gz', index=False, compression='gzip')

In [39]:
df = forecasts1819_all.copy()
df = df.sort_values(['Model','location','forecast_date', 'horizon', 'bin_start_incl','submission_date']).copy()
df['cum_val'] = df.groupby(['Model','location','horizon','forecast_date','submission_date'])['value'].cumsum()


result1819 = (df.groupby(['Model',"location", "horizon", "forecast_date", 'season', 'target_end_date',
                      'submission_date']).apply(get_23_quantiles_interp).reset_index()).drop(columns='level_7')


In [40]:
result1819.to_csv('../dat/ILI team forecasts/ILIforecasts_teams_quantiles_2018-19.csv.gz', index=False, compression='gzip')

In [41]:
df = forecasts1920_all.copy()
df = df.sort_values(['Model','location','forecast_date', 'horizon', 'bin_start_incl','submission_date']).copy()
df['value'] = df['value'].astype(float)
df['cum_val'] = df.groupby(['Model','location','horizon','forecast_date','submission_date'])['value'].cumsum()


result1920 = (df.groupby(['Model',"location", "horizon", "forecast_date", 'season', 'target_end_date',
                      'submission_date']).apply(get_23_quantiles_interp).reset_index()).drop(columns='level_7')


In [42]:
result1920.to_csv('../dat/ILI team forecasts/ILIforecasts_teams_quantiles_2019-20.csv.gz', index=False, compression='gzip')